# `synthmed` demo

A minimal notebook that exercises the `synthmed` package end-to-end.
It contains **no business logic** — every step delegates to the package.

All inputs now live under `inputs/`:

- `inputs/schemas/<cohort>/<year>/*.fts` — schema files (committed).
- `inputs/distributions/` — reference distributions (committed; see
  `docs/distributions/` for per-file provenance).
- `inputs/samples/` — CMS DE-SynPUF inpatient sample CSVs
  (~320 MB, **not committed**; lazy-downloaded on first run, or
  populate manually with `synthmed download-samples`).

In [ ]:
from pathlib import Path

from synthmed import GenerationConfig, load_distributions, run

REPO_ROOT = Path.cwd().parent

## 1. Configure the run

In [ ]:
config = GenerationConfig(
    data_root=REPO_ROOT / "inputs" / "schemas",
    distribution_dir=REPO_ROOT / "inputs" / "distributions",
    # First run will lazy-download the ~320 MB DE-SynPUF samples from CMS
    # into this directory (or set SYNTHMED_OFFLINE=1 to disable that and
    # populate the directory manually). See docs/distributions/medicare_sample_data.md
    sample_dir=REPO_ROOT / "inputs" / "samples",
    output_dir=REPO_ROOT / "output_dat_files",
    total_people=1000,
)
config

## 2. (Optional) inspect the reference distributions

Loading distributions explicitly lets us peek at them before running the
pipeline; `run(config)` would otherwise load them for us.

In [ ]:
dist = load_distributions(config.distribution_dir, config.sample_dir)
dist.state_error_medpar.head()

## 3. Run the full multi-year pipeline

Produces `.dat` files in `config.output_dir`, mirroring the
`<cohort>/<year>/` layout of `config.data_root`, and copies the matching
`.fts` schemas next to them.

In [ ]:
directory_map = run(config, dist=dist)
sorted(directory_map.keys())

## 4. Spot-check an output file

In [ ]:
first_year = sorted(directory_map.keys())[0]
out_dir = directory_map[first_year]["output"]
list(out_dir.iterdir())